## 0. Environment Configuration

All imports consolidated here. GPU acceleration is used when available.
Path configuration uses `pathlib` for portability.

In [ ]:
import datetime, os, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nmrglue as ng
from scipy.signal import find_peaks
from scipy.ndimage import maximum_filter, gaussian_filter
from scipy.spatial import KDTree
from scipy import stats
from scipy.optimize import lsq_linear
import torch
import models_RHUnet
from torch.utils.data import TensorDataset, DataLoader

torch.set_default_tensor_type(torch.FloatTensor)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def printbar():
    nowtime = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print("=" * 8 + nowtime)

# ---- Path configuration ----
NB_DIR = Path.cwd()
DATA_DIR = NB_DIR / "data" / "20240622insight_MCF_7_1"
FACTOR_DIR = NB_DIR / "factors"
FACTOR_DIR.mkdir(parents=True, exist_ok=True)

printbar()
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | Device: {device}")
print(f"Data dir: {DATA_DIR.name} (relative to notebook)")
print(f"Output dir: {FACTOR_DIR.name} (relative to notebook)")


## 1. Data Loading

Bruker NMR experiments are listed and sorted by experiment number.
NOESY (even experiment numbers from 4) and HSQC (odd numbers from 3)
spectra are loaded via `nmrglue`.

In [ ]:
# ---- Load Bruker data ----
exp_name = DATA_DIR.name
print(f"Experiment: {exp_name}")  # from data/

# Load NOESY spectra (expno 4, 6, ..., 32)
noesy_list = []
for expno in range(4, 33, 2):
    pdata_path = DATA_DIR / str(expno) / "pdata" / "1"
    try:
        dic, data = ng.bruker.read_pdata(str(pdata_path))
        noesy_list.append(data)
    except Exception:
        pass
print(f"Loaded {len(noesy_list)} NOESY spectra")

# Load HSQC spectra (expno 3, 5, ..., 31)
hsqc_list = []
for expno in range(3, 32, 2):
    pdata_path = DATA_DIR / str(expno) / "pdata" / "1"
    try:
        dic, data = ng.bruker.read_pdata(str(pdata_path))
        hsqc_list.append(data)
    except Exception:
        pass

printbar()
print(f"Loaded {len(hsqc_list)} HSQC spectra")
hsqc_array = np.array(hsqc_list) / hsqc_list[0].max()
hsqc_array.shape


## 2. Chemical Shift Axes

Define H1 and C13 chemical shift axes (ppm) for HSQC and NOESY spectra.
These are used throughout all subsequent plots. Index-based axes are also defined
for pixel-domain operations.

In [ ]:
printbar()

# ---- Spectral parameters ----
H1_FREQ    = 4.691           # ppm, H1 carrier
H1_SW_HSQC = 16.0216         # ppm, H1 sweep width (HSQC)
H1_SW_NOE  = 16.022          # ppm, H1 sweep width (NOESY)
C13_SW     = 79.9988         # ppm, C13 sweep width
C13_FREQ   = 40.0            # ppm, C13 carrier
N_H1_4K    = 4 * 1024
N_H1_64K   = 64 * 1024
N_C13      = 128

# ---- Chemical shift axes ----
h1_sw  = np.linspace(H1_FREQ + H1_SW_HSQC / 2, H1_FREQ - H1_SW_HSQC / 2, N_H1_4K)
c13_sw = np.linspace(C13_FREQ + C13_SW / 2,     C13_FREQ - C13_SW / 2,     N_C13)
noesy_sw = np.linspace(H1_FREQ + H1_SW_NOE / 2, H1_FREQ - H1_SW_NOE / 2,  N_H1_64K)

# ---- Index axes (pixel domain) ----
sw64, sw4 = np.linspace(0, N_H1_4K, N_H1_64K), np.linspace(0, N_H1_4K, N_H1_4K)

print(f"H1 axis:  {h1_sw.shape}  [{h1_sw[0]:.2f} .. {h1_sw[-1]:.2f}] ppm")
print(f"C13 axis: {c13_sw.shape}  [{c13_sw[0]:.1f} .. {c13_sw[-1]:.1f}] ppm")
print(f"NOESY axis: {noesy_sw.shape}")


## 3. NOESY Processing & Visualization

Baseline correction uses the first 30,000 points as a noise region, followed by
max-normalization. Four complementary views are shown.

In [ ]:
noesy_np = np.array(noesy_list)
noesy_np -= noesy_np[:,0:30000].min(1).reshape(-1,1)
noesy_np /= noesy_np.max()

printbar()
print(noesy_np.shape)


In [ ]:
plt.figure(figsize=(20,10))
plt.subplot(2,1,1)
plt.plot(noesy_np.T, alpha = 0.5)
plt.ylim(-0.1, 1.05)
plt.subplot(2,1,2)
plt.plot(noesy_np.T, alpha = 0.5)
plt.vlines(np.argmax(noesy_np[1]), -0.1, 1.1, linestyles='--',colors = 'r' )
plt.ylim(-0.1, 1.05)
plt.xlim(46000, 49000)
printbar()
plt.show()

plt.figure(figsize=(20,9))
plt.subplot(3,1,1)
plt.plot(noesy_sw, noesy_np.T )
plt.xlim(5.5, 3)
plt.ylim(-0.05, 0.7)
plt.subplot(3,1,2)
plt.plot(noesy_sw, noesy_np.T )
plt.xlim(2.5, 1.5)
plt.ylim(-0. , 0.25)
plt.subplot(3,1,3)
plt.plot(noesy_sw, noesy_np.T )
plt.xlim(1.5, 0.7)
plt.ylim(-0. , 1.05)
printbar()
plt.show()

plt.figure(figsize=(12,6))
for n,i in enumerate(noesy_np):
    plt.plot(noesy_sw,  i+n*0.3)
plt.xlim(4.5, 0.5)
plt.ylim(-0.1, 1.1+n*0.3)
ax = plt.gca()
ax.spines['left'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.yticks([])
plt.tick_params(labelsize=16)
printbar()
plt.show()

plt.figure(figsize=(10,4))
plt.plot(noesy_sw,  noesy_np[0])
plt.xlim(4.5, 0.5)
plt.ylim(-0.05, 0.8)
ax = plt.gca()
ax.spines['left'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.yticks([])
plt.tick_params(labelsize=16)
printbar()
plt.show()


## 4. HSQC Slice Extraction

Extract 1D slices at selected C13 positions from the HSQC pseudo-3D array.
Three groups are visualized: aliphatic (C13=9,41,63), anomeric (C13=77,81,92,96),
and other key peaks (C13=108,18).

In [ ]:
slice_9 = hsqc_array[:,9,:]
slice_41 = hsqc_array[:,41,:]
slice_63 = hsqc_array[:,63,:]
slice_77 = hsqc_array[:,77,:]
slice_81 = hsqc_array[:,81,:]
slice_92 = hsqc_array[:,92,:]
slice_96 = hsqc_array[:,96,:]
slice_108 = hsqc_array[:,108,:]
slice_18 = hsqc_array[:,18,:]
printbar()

In [ ]:
plt.figure(figsize=(16,6))
plt.subplot(1,3,1)
plt.plot( slice_9.T/slice_9.max(1), alpha = 0.5)
plt.xlim(2350, 2500)
plt.title('slice_9')
plt.subplot(1,3,2)
plt.plot( slice_41.T/slice_41.max(1), alpha = 0.5)
plt.xlim(2350, 2500)
plt.title('slice_41')
plt.subplot(1,3,3)
plt.plot( slice_63.T/slice_63.max(1), alpha = 0.5)
plt.xlim(2350, 2500)
plt.title('slice_63')
printbar()
plt.show()

plt.figure(figsize=(20,6))
plt.subplot(1,4,1)
plt.plot( slice_81.T/slice_81.max(1), alpha = 0.5)
plt.ylim(-0.1, 1.1)
plt.xlim(2800, 3000)
plt.title('slice_81')
plt.subplot(1,4,2)
plt.plot( slice_92.T/slice_92[:,2800: 2950].max(1), alpha = 0.5)
plt.ylim(-0.1, 1.1)
plt.xlim(2800, 3000)
plt.title('slice_92')
plt.subplot(1,4,3)
plt.plot( slice_96.T/slice_96.max(1), alpha = 0.5)
plt.ylim(-0.1, 1.1)
plt.xlim(2800, 3000)
plt.title('slice_96')
plt.subplot(1,4,4)
plt.plot( slice_77.T/slice_77[:,2800: 2950].max(1), alpha = 0.5)
plt.ylim(-0.1, 1.1)
plt.xlim(2800, 3000)
plt.title('slice_77')
printbar()
plt.show()

plt.figure(figsize=(16,6))
plt.subplot(1,2,1)
plt.plot( slice_108.T/slice_108.max(1), alpha = 0.5)
plt.ylim(-0.1, 1.1)
plt.xlim(1850, 2050)
plt.title('slice_108')
plt.subplot(1,2,2)
plt.plot( slice_18.T/slice_18.max(1), alpha = 0.5)
plt.ylim(-0.1, 1.1)
plt.xlim(2100, 2450)
plt.title('slice_18')
printbar()
plt.show()


## 5. HSQC 2D Contour Maps

Contour plots of the first HSQC spectrum with both pixel and ppm axes.
Includes zoomed views of the aliphatic and anomeric regions.
The `CONTOUR_LEVELS` variable is reused throughout the notebook.

In [ ]:
hsqc_4k_128_2D = hsqc_list[0]
hsqc_4k_128_2D /= hsqc_4k_128_2D.max()
edlev = [i for i in np.linspace(0.05*1, 1, 100)]


plt.figure(figsize=(10, 4))
hsqc_4k_128_2D = hsqc_array[0]
edlev = [i for i in np.linspace(0.05*1, 1, 100)]
plt.subplot(1,2,1)
plt.contour(hsqc_4k_128_2D, edlev)
plt.subplot(1,2,2)
plt.contour(h1_sw, c13_sw, hsqc_4k_128_2D, edlev)
plt.xlim(4.691+16.0216/2,  4.691-16.0216/2)
plt.ylim(40+79.9988/2, 40-79.9988/2)
printbar()
plt.show()

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.contour(hsqc_4k_128_2D, edlev)
plt.xlim(2100, 3100)
plt.ylim(0, 120)
plt.subplot(1,2,2)
plt.contour(h1_sw,c13_sw, hsqc_4k_128_2D, edlev)
plt.xlim(h1_sw[2100], h1_sw[3100])
plt.ylim(c13_sw[0], c13_sw[120])
printbar()
plt.show()

plt.figure(figsize=(10, 4))
plt.subplot(1,2,1)
plt.contour(hsqc_4k_128_2D, edlev)
plt.xlim(2876, 2976)
plt.ylim(73, 100)
plt.subplot(1,2,2)
plt.contour(h1_sw,c13_sw, hsqc_4k_128_2D, edlev)
plt.xlim(h1_sw[2876], h1_sw[2976])
plt.ylim(c13_sw[73], c13_sw[100])
printbar()
plt.show()



## 6. Peak Detection

Automated peak picking on the HSQC 2D spectrum:
1. Gaussian smoothing (sigma=0.5)
2. Local maximum filter (5x5 window)
3. Intensity threshold (>0.05 of max)
4. KDTree-based deduplication (minimum distance = 5 pixels)

In [ ]:
def detect_real_peaks(
    image,
    threshold=0.15,
    sigma=0.5,
    window_size=3,
    min_distance=5
):

    img_smooth = gaussian_filter(image, sigma=sigma)
    local_max = maximum_filter(img_smooth, size=window_size) == img_smooth
    peaks_mask = local_max & (img_smooth > threshold)
    y_candidates, x_candidates = np.where(peaks_mask)
    if len(x_candidates) == 0:
        return np.array([]), np.array([])

    coords = np.column_stack((x_candidates, y_candidates))
    intensities = img_smooth[y_candidates, x_candidates]
    order = intensities.argsort()[::-1]
    coords = coords[order]
    keep = np.ones(len(coords), dtype=bool)
    tree = KDTree(coords)
    for i in range(len(coords)):
        if keep[i]:
            neighbors = tree.query_ball_point(coords[i], r=min_distance)
            for j in neighbors:
                if j > i:
                    keep[j] = False
    x_keep = coords[keep, 0]
    y_keep = coords[keep, 1]

    return y_keep, x_keep

hsqc_4k_2D = hsqc_array[0]
y, x = detect_real_peaks(
    hsqc_4k_2D,
    threshold=0.05,
    sigma=0.5,
    window_size=5,
    min_distance=5
)
plt.figure(figsize=(10,4))
hsqc_4k_128_2D = hsqc_array[0]
edlev = [i for i in np.linspace(0.05*1, 1, 100)]
plt.subplot(1,2,1)
plt.contour(h1_sw, c13_sw, hsqc_4k_128_2D, edlev)
plt.scatter(h1_sw[x], c13_sw[y], color='red', s=2)
plt.xlim(4.5, 0.5)
plt.ylim(40+79.9988/2, 40-79.9988/2)
plt.title(f"Detected Peaks: {len(x)}@threshold={0.05}", fontsize=14)
plt.subplot(1,2,2)
plt.scatter(h1_sw[x], c13_sw[y], color='red', s=2)
plt.xlim(4.5, 0.5)
plt.ylim(40+79.9988/2, 40-79.9988/2)
plt.title(f"Detected Peaks: {len(x)}@threshold={0.05}", fontsize=14)
plt.show()
print(f"\n✅ real peak count：{len(x)}")


In [ ]:
plt.figure(figsize=(15, 4))
plt.subplot(1,2,1)
plt.plot(noesy_sw, noesy_np[0] )
plt.plot( noesy_sw[x*16], noesy_np[0][x*16]+0.02, "x")
plt.xlim(4.5,  0.5)
plt.ylim(-0. , 0.8)
plt.subplot(1,2,2)
plt.plot(noesy_sw, noesy_np[0] )

for xi in x:
    pos_x = noesy_sw[xi*16]
    pos_y = noesy_np[0][xi*16]
    plt.annotate('p', xy=(pos_x, pos_y), xytext=(pos_x, pos_y+0.05),
                 arrowprops=dict(arrowstyle='->', color='red'))
plt.xlim(2.75,  0.75)
plt.ylim(-0. , 0.8)
plt.show()
printbar()

plt.figure(figsize=(20,9))
plt.subplot(2,3,1)
plt.plot(noesy_sw, noesy_np.T )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.xlim(1.1, 0.7)
plt.ylim(-0. , 0.4)
plt.subplot(2,3,2)
plt.plot(noesy_sw, noesy_np.T )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.xlim(1.4, 1.1)
plt.ylim(-0. , 1.05)
plt.subplot(2,3,3)
plt.plot(noesy_sw, noesy_np.T )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.xlim(1.8, 1.6)
plt.ylim(-0.00, 0.2)
plt.subplot(2,3,4)
plt.plot(noesy_sw, noesy_np.T )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.xlim(2.2, 2.)
plt.ylim(-0.0, 0.3)
plt.subplot(2,3,5)
plt.plot(noesy_sw, noesy_np.T )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.xlim(2.35, 2.15)
plt.ylim(-0.0, 0.2)
plt.subplot(2,3,6)
plt.plot(noesy_sw, noesy_np.T )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.xlim(2.6, 2.3)
plt.ylim(-0.0, 0.3)
printbar()
plt.show()

plt.figure(figsize=(10,6))
plt.subplot(2,3,1)
plt.plot(noesy_sw, noesy_np.T )
plt.xlim(3.3, 3)
plt.ylim(-0. , 0.6)
plt.subplot(2,3,2)
plt.plot(noesy_sw, noesy_np.T )
plt.xlim(5.4, 5)
plt.ylim(-0. , 0.4)
printbar()
plt.show()

plt.figure(figsize=(12,6))
plt.subplot(2,4,1)
plt.plot(noesy_sw, noesy_np.T )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.xlim(1.1, 0.7)
plt.ylim(-0. , 0.3)
plt.subplot(2,4,2)
plt.plot(noesy_sw, noesy_np.T )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.xlim(1.4, 1.1)
plt.ylim(-0. , 1.05)
plt.subplot(2,4,3)
plt.plot(noesy_sw, noesy_np.T )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.xlim(1.8, 1.6)
plt.ylim(-0.00, 0.15)
plt.subplot(2,4,4)
plt.plot(noesy_sw, noesy_np.T )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.xlim(2.2, 2.)
plt.ylim(-0.0, 0.25)
plt.subplot(2,4,5)
plt.plot(noesy_sw, noesy_np.T )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.xlim(2.35, 2.15)
plt.ylim(-0.0, 0.15)
plt.subplot(2,4,6)
plt.plot(noesy_sw, noesy_np.T )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.xlim(2.6, 2.3)
plt.ylim(-0.0, 0.25)
plt.subplot(2,4,7)
plt.plot(noesy_sw, noesy_np.T )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.xlim(3.3, 3)
plt.ylim(-0. , 0.6)
plt.subplot(2,4,8)
plt.plot(noesy_sw, noesy_np.T )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.xlim(5.4, 5)
plt.ylim(-0. , 0.4)
printbar()
plt.show()


## 7. NOESY Peak Analysis: Begin vs End

Compare the first and last NOESY spectra to identify time-dependent changes.
HSQC-detected peak positions are overlaid as reference markers.

In [ ]:
plt.figure(figsize=(10, 6))
plt.subplot(2,1,1)
plt.plot( noesy_np.T, alpha = 0.5)
plt.ylim(-0.1, 1.1)
plt.xlim(10000, 60000)
plt.subplot(2,2,3)
plt.plot( noesy_np.T, alpha = 0.5)
plt.vlines(np.argmax(noesy_np[1][ :31000]), -0.1, 1.1, linestyles='--',colors = 'r' )
plt.ylim(-0.05, 0.4)
plt.xlim(30000, 31000)
plt.subplot(2,2,4)
plt.plot( noesy_np.T, alpha = 0.5)
plt.vlines(np.argmax(noesy_np[1]), -0.1, 1.1, linestyles='--',colors = 'r' )
plt.ylim(-0.1, 1.1)
plt.xlim(46000, 49000)
printbar()
plt.show()

plt.figure(figsize=(12, 6))
plt.subplot(2,1,1)
plt.plot(noesy_np.T, alpha = 0.5)
plt.ylim(-0.02, 0.5)
plt.vlines(np.argmax(noesy_np[0,30000: 31000])+30000, -0.1, 1.1, linestyles='--',colors = 'r' )
plt.xlim(30000, 31000)
plt.subplot(2,1,2)
plt.plot(noesy_np.T, alpha = 0.5)
plt.vlines(np.argmax(noesy_np[0]), -0.1, 1.1, linestyles='--',colors = 'r' )
plt.ylim(-0.1, 1.1)
plt.xlim(46000, 49000)
printbar()
plt.show()

plt.figure(figsize=(10, 6))
plt.subplot(2,1,1)
plt.plot(noesy_np[0], alpha = 0.5, label = 'begin')
plt.plot(noesy_np[-1], alpha = 0.5, label = 'end')
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.ylim(-0.05, 1.05)
plt.xlim(33000, 49000)
plt.legend()
plt.subplot(2,1,2)
plt.plot(noesy_np[0], alpha = 0.5, label = 'begin')
plt.plot(noesy_np[-1], alpha = 0.5, label = 'end')
plt.vlines(np.argmax(noesy_np[0]), -0.1, 1.1, linestyles='--',colors = 'r' )
plt.plot(noesy_sw[x*16], noesy_np[0][x*16], 'ro')
plt.ylim(-0.05, 1.05)
plt.xlim(46000, 49000)
plt.legend()
printbar()
plt.show()

plt.figure(figsize=(20, 10))
plt.subplot(3,1,1)
plt.plot(noesy_sw, noesy_np[0], label = 'begin')
plt.plot(noesy_sw, noesy_np[-1], label = 'end')
plt.plot(noesy_sw[x*16],  np.maximum(noesy_np[0][x*16],noesy_np[-1][x*16]) , 'o')
plt.xlim(5.5, 3)
plt.ylim(-0.05, 0.7)
plt.legend()
plt.subplot(3,1,2)
plt.plot(noesy_sw, noesy_np[0], label = 'begin')
plt.plot(noesy_sw, noesy_np[-1], label = 'end')
plt.plot(noesy_sw[x*16], np.maximum(noesy_np[0][x*16],noesy_np[-1][x*16]), 'o')
plt.xlim(2.5, 1.5)
plt.ylim(-0. , 0.25)
plt.legend()
plt.subplot(3,1,3)
plt.plot(noesy_sw, noesy_np[0], label = 'begin')
plt.plot(noesy_sw, noesy_np[-1], label = 'end')
plt.plot(noesy_sw[x*16], np.maximum(noesy_np[0][x*16],noesy_np[-1][x*16]), 'o', markersize=6)
plt.xlim(1.5, 0.7)
plt.ylim(-0. , 1.05)
plt.legend()
printbar()
plt.show()


## 8. Peak Intensity Time Series

Manually curated peak list (46 peaks) with pixel coordinates.
Intensity is extracted as the maximum in a 3x3 window around each peak
for every time point. Statistical analysis identifies significantly changing peaks
(top 10% by standard deviation).

In [ ]:
x_arr   = np.array([2918, 2377, 2296, 2361, 2251, 2419, 2910, 2286, 2266, 2344, 2270,
        2299, 2427, 2733, 1910, 2677, 3022, 2845, 3003, 2918, 2924, 2983,
        2705, 2289, 2635, 2647, 2996, 2991, 2475, 2375, 1891, 2197, 2623,
        2808, 2871, 3009, 2208, 2182, 2327, 2655, 2814, 2282, 2891, 3010,
        2338, 2596])
y_arr   = np.array([ 81,  16,  30,   6,  30,   9,  96,  31,  31,  13,  13,  11,  41,
         84, 108,  74, 106,  88,  93,  92,  77,  99,  85,  40,  80,  74,
        101, 104,  65,   6,  49,  18,  78,  85, 102, 110,  29,  35,  31,
         74,  86,  58, 101,  99,  61,  78])


print("Peak coordinate pairs (x, y):")
for xi, yi in zip(h1_sw[x_arr], c13_sw[y_arr]):
    print(f"({np.round(xi, 2)}, {np.round(yi, 2)})")

In [ ]:
all_peak_int = []
for h,c in zip(x_arr,y_arr):
    peak_int = []
    for hsqc in hsqc_array:
        peak_int.append(hsqc[c-2:c+2,h-2:h+2].max())
    all_peak_int.append(peak_int)

len(all_peak_int), len(peak_int)

In [ ]:
data = np.array(all_peak_int)
x0 = np.arange(data.shape[1])

std_vals = data.std(axis=1)
std_threshold = np.percentile(std_vals, 90)
significant_peaks = std_vals > std_threshold

print(f"Total peaks: {len(data)}")
print(f"Significantly changing peaks: {np.sum(significant_peaks)}")
print(f"Indices of these peaks: {np.where(significant_peaks)[0]}")

significant_labels = y_arr[significant_peaks]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

for row in data:
    ax1.plot(x0, row, color='#e0e0e0', alpha=0.2, linewidth=0.8)

colors_light = ['#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5', '#c49c94']
for i, row in enumerate(data[significant_peaks]):
    ax1.plot(x0, row, color=colors_light[i % len(colors_light)], alpha=0.6, linewidth=1.2)

ax1.set_title("Intensity Changes of All 55 Peaks", fontsize=14, fontweight='bold')
ax1.set_xlabel("Time / Condition Index", fontsize=12)
ax1.set_ylabel("Intensity", fontsize=12)
ax1.grid(alpha=0.3, linestyle='--')

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
for i, row in enumerate(data[significant_peaks]):
    ax2.plot(x0, row, color=colors[i % len(colors)],
             alpha=1.0, linewidth=2.5, label=f'{significant_labels[i]}')

ax2.set_title(f"Significantly Changing Peaks (n={np.sum(significant_peaks)})",
              fontsize=14, fontweight='bold')
ax2.set_xlabel("Time / Condition Index", fontsize=12)
ax2.set_ylabel("Intensity", fontsize=12)
ax2.grid(alpha=0.3, linestyle='--')
plt.legend()

plt.tight_layout()
plt.show()


## 9. Super-Resolution Model: 4K HSQC to 4K NOESY

A **RHUnet** (Recursive Half-UNet) neural network is trained to map
HSQC-derived 1D projections to their corresponding NOESY traces.
This enables deconvolution of crowded spectral regions.

### 9a. Data Alignment Check

Verify that HSQC and NOESY traces align at the glucose H1 peak (5.23 ppm, C13=108).

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(noesy_sw[::16], hsqc_4k_128_2D[108], label='glu')
plt.plot(noesy_sw, noesy_np[0], label = 'noesy_64k')
plt.plot(noesy_sw[::16], noesy_np[0,::16], label = 'noesy_4k')
plt.plot(np.zeros_like(hsqc_4k_128_2D[108]), linestyle='--',c='k')
plt.xlim(5.4, 5)
plt.ylim(-0.05, 1)
plt.legend()
plt.show()

gluc_5p23 = hsqc_4k_128_2D[108]
plt.figure(figsize=(12,5))
plt.subplot(1,3,1)
plt.plot(gluc_5p23)
plt.plot(np.zeros_like(gluc_5p23), linestyle='--',c='k')
plt.xlim(1000, 3000)
plt.subplot(1,3,2)
plt.plot(gluc_5p23 )
plt.xlim(1000, 3000)
plt.subplot(1,3,3)
plt.plot(gluc_5p23)
plt.plot(np.zeros_like(gluc_5p23), linestyle='--',c='k')
plt.xlim(1800, 2000)
plt.show()


### 9b. Sum Spectrum Construction

In [ ]:
peaks_list = []
spe_data = 0
for peak_C13 in list(set(y) ):
    data_64 =  hsqc_4k_128_2D[peak_C13]

    height = 0.05
    peaks, _ = find_peaks(data_64, height=height)
    peaks_list += list(peaks)
    spe_data += data_64

len(peaks_list), spe_data.shape


### 9c. Template Construction

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(spe_data/2.5, label = 'spe_data')
plt.plot(noesy_np[0,::16], label = 'noesy_np[0]', alpha = 0.5)
plt.xlim(30000/16, 50000/16)
plt.ylim(-0.1, 1.5)
plt.legend()
plt.show()

glu_remove =   gluc_5p23
plt.figure(figsize=(12, 4))
plt.plot(spe_data[1900*1-10 :1925*1 ]/spe_data[1900*1 :1925*1 ].max(), label ='raw_glu')
plt.plot(glu_remove[1900*1-10 :1925*1 ]/glu_remove[1900*1 :1925*1 ].max(), label ='remove_glu')
plt.plot(noesy_np[0,::16][1900*1-10 :1925*1 ]/noesy_np[0,::16][ 1900*1 :1925*1 ].max(), label ='noesy')
plt.plot(np.zeros_like(spe_data[1900*1-10  :1925*1 ]), "--", color="gray")
plt.legend()
plt.show()

sim_data_f = np.zeros(spe_data.shape)
per_data_l = np.zeros(spe_data.shape)
sim_template = (glu_remove[1900*1 :1925*1 ]-glu_remove[1900*1 :1925*1 ].min())/glu_remove[1900*1 :1925*1 ].max()
per_template = (noesy_np[0,::16][1900*1 :1925*1 ]-noesy_np[0,::16][1900*1 :1925*1 ].min())/noesy_np[0,::16][1900*1 :1925*1 ].max()

for i in set(peaks_list):
    sim_data_f[i-12 : i+13 ] += sim_template*(spe_data[i] )
    per_data_l[i-12 : i+13 ] += per_template*(spe_data[i] )
sim_data_f[1900*1 :1925*1 ] = sim_template
per_data_l[1900*1 :1925*1 ] = per_template
scal = max(sim_data_f.max(), per_data_l.max())
X = sim_data_f/scal
y = per_data_l/scal
X = X.astype(np.float32)
y = y.astype(np.float32)
print(scal, X.max(), y.max())


### 9d. Template Visualization

In [ ]:
plt.figure(figsize = (12,5))
plt.plot(X, label ='X feature' )
plt.plot(y, label ='y label' )
plt.plot(np.zeros_like(X), "--", color="gray")
plt.vlines(1900*1 , -0.1, 1.1, linestyles='--',colors = 'r' )
plt.vlines(1925*1 , -0.1, 1.1, linestyles='--',colors = 'r' )
plt.xlim(1900*1 -12, 1925*1 + 13)
plt.ylim(-0.01,0.25)
plt.legend()
plt.show()

plt.figure(figsize= (16,5))
plt.subplot(1,4,(1, 3))
plt.plot(sim_data_f, label ='sim_data_f')
plt.plot(per_data_l, label ='per_data_l', alpha = 0.5)
plt.xlim(30000/16,50000/16)
plt.legend()
plt.subplot(1,4,4)
plt.plot(sim_data_f, label ='sim_data_f')
plt.plot(per_data_l, label ='per_data_l', alpha = 0.5)
plt.xlim(37200/16, 39000/16)
plt.legend()
plt.show()

plt.figure(figsize= (12, 5))
plt.subplot(1,4,(1, 2))
plt.plot(sim_data_f, label ='sim_data_f')
plt.plot(per_data_l, label ='per_data_l', alpha = 0.5)
plt.xlim(30000/16,35000/16)
plt.legend()
plt.subplot(1,4,(3,4))
plt.plot(sim_data_f, label ='sim_data_f')
plt.plot(per_data_l, label ='per_data_l', alpha = 0.5)
plt.xlim(37200/16, 39000/16)
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.subplot(1,4,(1, 3))
plt.plot(sim_data_f/15 , label = 'data')
plt.plot(noesy_np[0,::16], label = 'noesy_np[0]')
plt.plot(per_data_l/15+0.1 , label = 'target')
plt.xlim(46000/16, 47500/16)
plt.ylim(-0.05,1.25)
plt.legend()
plt.subplot(1,4,4)
plt.plot(sim_data_f/15 , label = 'data')
plt.plot(noesy_np[0,::16], label = 'noesy_np[0]')
plt.plot(per_data_l/15+0.1 , label = 'target')
plt.xlim(30000/16, 31000/16)
plt.ylim(-0.05,1.25)
plt.legend()
plt.show()

plt.figure(figsize= (12,5))
height = 0.10
peaks, _ = find_peaks(spe_data, height=height)
plt.plot(spe_data, label = 'spe_data')
plt.plot(peaks_list, spe_data[peaks_list], "x")
plt.plot(np.zeros_like(spe_data) + height, "--", color="gray")
plt.plot(noesy_np[0,::16]*5, label = 'noesy_np[0]', alpha = 0.5)
plt.ylim(-0.1,1.05*2.5)
plt.xlim(30000/16, 50000/16)
plt.legend()
plt.show()



### 9e. Model Training

In [ ]:
ti = time.time()
output_list = []
v_cut = 0.01

new_dataset = TensorDataset(torch.tensor(X).reshape(1,1,-1), torch.tensor(y).reshape(1,1,-1))
train_loader = DataLoader(dataset=new_dataset, batch_size=1)

net = models_RHUnet.uNet8()
model = net.to(device)
criterion = models_RHUnet.losss
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
per_fid = per_data_l/per_data_l.max()
per_fid = torch.tensor(per_fid).reshape(1,1,-1).to(device)
data_14_2 = spe_data/spe_data[1900*1 :1925*1 ].max()
v_data_14 = torch.tensor(data_14_2).reshape(1,1,-1).to(torch.float32).to(device)
loss_list = []
v_list_tep = []

for epoch in range(0, 2*1000):
    for data, target in train_loader:
        X = data.to(device)
        y = (target).to(device)
        model.train()
        optimizer.zero_grad()
        output = model(X )
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        loss_list.append(loss.data)

    if epoch % 50 == 0:
        optimizer.param_groups[0]['lr']*=0.95

    output_n = model(v_data_14)
    val_loss = models_RHUnet.losss_stop(output_n[:,:,1900*1 :1900*1 +25]/output_n[:,:,1900*1 :1900*1 +25].max(), per_fid[:,:,1900*1 :1900*1 +25])
    v_list_tep.append(val_loss.item())

    if  val_loss <= v_cut:
        print(val_loss.item(), 'GOOD! Training Done')
        output_list.append(output_n.squeeze().cpu().detach())
        break

    print('Epoch[{}/{}/{}], \tloss: {:.6f}, \tval_loss: {:.6f}, '.format('test', 2000,  epoch, loss.data*10000, val_loss))
    if np.isnan(val_loss.item()):
        print('NAN')
        break

end_time = time.time()
print('total time:{:.3f}s '.format(end_time-ti ))
#  85.133s     85.082s ,   77.474s

plt.figure(figsize=(10,4))
plt.plot(output_n.detach().cpu().numpy().reshape(-1,1), label = 'output_n')
plt.plot(spe_data, label = 'input' )
plt.plot(noesy_np[0,::16 ]*3+.1, label = 'noesy_np[0] 4k')
plt.xlim(1900*1 -64, 1925*1 +64)
plt.ylim(-0.05, 1.5)
plt.vlines(1900*1 +80, -0.1, 1.1, linestyles='--',colors = 'r' )
plt.vlines(1900*1 +280, -0.1, 1.1, linestyles='--',colors = 'r' )
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(noesy_sw[::16] ,(output_n/3).detach().cpu().numpy().reshape(-1,1), label = 'output')
plt.plot(noesy_sw[::16] , noesy_np[0,::16]*1.2, label = 'noesy_4k')
plt.plot(noesy_sw[::16] , spe_data/4. , label = 'spe_data')
plt.xlim(1.4, 1.1)
plt.ylim(-0.05, 1.05)
plt.legend()
plt.show()


### 9f. Inference: Generate scale_all_np

In [ ]:
scale_all_n = []
model.eval()
start_time = time.time()
with torch.no_grad():
    hsqc_64k = hsqc_4k_128_2D
    for spec_data in hsqc_64k:
        output_n_tep = model(torch.from_numpy(spec_data).reshape(1,1,-1).to(torch.float32).to(device))
        scale_all_n.append(output_n_tep.cpu().numpy().reshape(-1,1))

end_time = time.time()
elapsed_time = end_time - start_time
print(f'cost time {elapsed_time:.6f} secs')
scale_all_np = np.array(scale_all_n).reshape(128, -1)
print(scale_all_np.shape)
# cost time 4.691608 secs 4k


## 10. Scale Output Comparison

Compare the model's super-resolved output (`scale_all_np`) against the raw HSQC
to validate the enhancement quality. Includes 2D contour comparisons and
single-slice overlays.

In [ ]:
scale_all_np =  scale_all_np/scale_all_np.max() # np.load('output_n_scale_all_np.npy')
print(scale_all_np.shape, scale_all_np.max())

plt.figure(figsize=(20,10))
plt.subplot(1,2,1)
plt.contour( hsqc_4k_128_2D,  edlev)
plt.xlim(2100, 3100)
plt.ylim(0, 120)
plt.subplot(1,2,2)
plt.contour(h1_sw,c13_sw,   hsqc_4k_128_2D,  edlev)
plt.xlim(h1_sw[2100],  h1_sw[3100])
plt.ylim(c13_sw[0], c13_sw[120])
plt.show()

plt.figure(figsize=(20,10))
plt.subplot(1,2,1)
plt.contour(scale_all_np, edlev)
plt.xlim(2100*1 , 2500*1 )
plt.ylim(0, 70)
plt.title('scale')
plt.subplot(1,2,2)
plt.contour(hsqc_4k_128_2D, edlev)
plt.xlim(2100, 2500)
plt.ylim(0, 70)
plt.title('raw')
printbar()
plt.show()


sw64_2, sw4 = np.linspace(0,4*1024,64*1024), np.linspace(0,4*1024,4*1024)
printbar()
sw64_2.shape, sw4.shape




## 11. LSQ Deconvolution

Non-negative least-squares fitting decomposes the NOESY spectrum into contributions
from individual HSQC-detected metabolites. Four spectral regions are fit independently.

### Helper Functions

In [ ]:
def lsq_fit_peaks(noesy_4k, scales, peak_windows, baseline_region=None, constraints=None):
    """
    Generic non-negative LSQ fitting of NOESY spectra using HSQC scale components.

    Parameters
    ----------
    noesy_4k : ndarray (N_time, 4096)
        NOESY spectra at 4K resolution, one row per time point.
    scales : list of ndarray
        HSQC scale components, each shape (4096,).
    peak_windows : list of slice
        Slices defining fitting regions.
    baseline_region : slice, optional
        Region for baseline estimation.
    constraints : tuple (C, d), optional
        Inequality constraints C @ x >= d.

    Returns
    -------
    coeffs : ndarray (N_time, N_scales)
        Fitted coefficients for each scale component across time.
    """
    def _extract(data, windows):
        return np.concatenate([data[w] for w in windows])

    A = np.array([_extract(s, peak_windows) for s in scales]).T
    W = np.diag(np.ones(A.shape[0]))
    A_w = W @ A

    if baseline_region is None:
        bl = slice(min(w.start for w in peak_windows),
                   max(w.stop  for w in peak_windows))
    else:
        bl = baseline_region

    n_scales = len(scales)
    coeffs = np.zeros((len(noesy_4k), n_scales))

    for t, spectrum in enumerate(noesy_4k):
        baseline = spectrum[bl].min()
        b = W @ _extract(spectrum - baseline, peak_windows)

        if constraints is not None:
            C_mat, d_vec = constraints
            A_aug = np.vstack([A_w, C_mat])
            b_aug = np.concatenate([b, d_vec])
            res = lsq_linear(A_aug, b_aug, bounds=(0.0, np.inf), method="bvls")
        else:
            res = lsq_linear(A_w, b, bounds=(0.0, np.inf), method="bvls")

        coeffs[t] = res.x

    return coeffs

def append_csv_columns(csv_path, col_dict):
    """Append columns to an existing CSV, or create it if new."""
    df_new = pd.DataFrame(col_dict)
    if Path(csv_path).exists():
        df_old = pd.read_csv(csv_path)
        df_new = pd.concat([df_old, df_new], axis=1)
    df_new.to_csv(csv_path, index=False)
    print(f"  Saved columns {list(col_dict.keys())} -> {Path(csv_path).name}")

print("LSQ helpers defined.")


### 11a. Glucose Peak @ 5.23 ppm (C13=108)

Single-component fit using the glucose H1 peak as the reference scale component.

In [ ]:
scale_108 = scale_all_np.reshape(128, -1)[108]
printbar()

plt.figure(figsize=(8, 4))
plt.plot( noesy_np[0,::16 ]+0, label = 'noesy_0')
plt.plot(  scale_108*0.7, label = 'glucose')
plt.xlim(30000/16, 31000/16)
plt.ylim(-0.05, 1)
plt.legend()
plt.show()

peak_slice = slice(1905, 1915)   # == glucose peak region ==
def extract_peaks(data):
    return data[peak_slice]
scales = [scale_108]# reference spectrum
A = np.array([extract_peaks(s) for s in scales]).T  # shape (N, 1)

weights = np.ones(A.shape[0])#
W = np.diag(weights)
A_weighted = W @ A

bounds = (0.0, np.inf)# Bounds: coefficients ≥0
glucose_list = []
for i in noesy_np[:,::16 ]:
    b = extract_peaks(i)
    b_weighted = W @ b
    # Non-negative least squares
    res = lsq_linear(A_weighted, b_weighted, bounds=bounds, method="bvls")
    fac = res.x[0]
    glucose_list.append(fac)

print(glucose_list, len(glucose_list))


In [ ]:
plt.figure(figsize=(20,6))
plt.subplot(1,2,1)
plt.plot(h1_sw, noesy_np[0,::16 ]-noesy_np[0,::16 ][1900:1940].min(), label = 'noesy_0_4k')
plt.plot(h1_sw,  scale_108*glucose_list[0], label = '108')
plt.xlim(h1_sw[1900], h1_sw[1940])
plt.ylim(-0.05, 0.6)
plt.legend()
plt.subplot(1,2,2)
plt.plot(h1_sw, noesy_np[0,::16 ]-noesy_np[0,::16 ][1900:1940].min(), label = 'noesy_0_4k')
plt.plot(h1_sw,  scale_108*glucose_list[0], label = '108')
plt.xlim(h1_sw[1900], h1_sw[1940])
plt.ylim(-0.05, 0.6)
plt.legend()
plt.show()
# 108

plt.figure(figsize=(20,6))
plt.subplot(1,2,1)
plt.plot(h1_sw, noesy_np[-1,::16 ]-noesy_np[-1,::16 ][1900:1940].min(), label = 'noesy_-1_4k')
plt.plot(h1_sw,  scale_108*glucose_list[-1], label = '108')
plt.xlim(h1_sw[1900], h1_sw[1940])
plt.ylim(-0.05, 0.6)
plt.legend()
plt.subplot(1,2,2)
plt.plot(h1_sw, noesy_np[-1,::16 ]-noesy_np[-1,::16 ][1900:1940].min(), label = 'noesy_-1_4k')
plt.plot(h1_sw,  scale_108*glucose_list[-1], label = '108')
plt.xlim(h1_sw[1900], h1_sw[1940])
plt.ylim(-0.05, 0.6)
plt.legend()
plt.show()
# 108


In [ ]:
factor_path = FACTOR_DIR / f"{exp_name}_scale.csv"
factor_HSQC_path = FACTOR_DIR / f"{exp_name}_HSQC.csv"

df = pd.DataFrame({
    "glucose_108": glucose_list,   #scale_108
})
df.to_csv(str(factor_path), index=False, mode='a')

df_HSQC = pd.DataFrame({
    "glucose_HSQC": all_peak_int[14],   #scale_108
})
df_HSQC.to_csv(str(factor_HSQC_path), index=False, mode='a')


In [ ]:
# factor_HSQC_path

### 11b. 3.15-3.30 ppm Region (C13=9, 41, 63)

Three-component fit in the crowded aliphatic region.

In [ ]:
scale_9 = scale_all_np.reshape(128, -1)[9]
scale_41 = scale_all_np.reshape(128, -1)[41]
scale_63 = scale_all_np.reshape(128, -1)[63]
scale_9.shape, scale_41.shape, scale_63.shape

plt.figure(figsize=(10,6))
plt.subplot(1,2,1)
plt.contour(hsqc_4k_128_2D, edlev)
plt.xlim(2400, 2443)
plt.ylim(5, 70)
plt.subplot(1,2,2)
plt.contour(h1_sw,c13_sw, hsqc_4k_128_2D, edlev)
plt.xlim(h1_sw[2400], h1_sw[2443])
plt.ylim(c13_sw[5], c13_sw[70])
printbar()
plt.show()

peak_windows = [slice(2405, 2440)  ]
def extract_peaks(data, windows):
    return np.concatenate([data[w] for w in windows])

scales = [scale_9, scale_41, scale_63 ]
A = np.array([extract_peaks(s, peak_windows) for s in scales]).T

weights = np.ones(A.shape[0])
W = np.diag(weights)
A_weighted = W @ A

bounds = (0.0, np.inf)

fac1_list, fac2_list, fac3_list = [], [], []

for i in noesy_np[:, ::16]: #:50
    b = extract_peaks(i-i[2400:2440].min(), peak_windows)
    b_weighted = W @ b
    res = lsq_linear(A_weighted, b_weighted, bounds=bounds, method="bvls")
    coeffs = res.x

    fac1_list.append(coeffs[0])
    fac2_list.append(coeffs[1])
    fac3_list.append(coeffs[2])

printbar()
print(len(fac1_list))


In [ ]:
plt.figure(figsize=(20,6))
plt.subplot(1,2,1)
plt.plot( noesy_np[-1,::16]-noesy_np[0,::16 ][2400:2440].min(), label = 'noesy_-1_4k')
plt.plot(  scale_9*fac1_list[-1], label = '9')
plt.plot(  scale_41*fac2_list[-1], label = '41')
plt.plot(  scale_63*fac3_list[-1], label = '63')
plt.xlim(38400/16, 39100/16)
plt.ylim(-0.05, 0.6)
plt.legend()
plt.subplot(1,2,2)
plt.plot( noesy_np[-1,::16]-noesy_np[0,::16 ][2400:2440].min(), label = 'noesy_-1_4k')
plt.plot(  scale_9*fac1_list[-1] + scale_41*fac2_list[-1]+scale_63*fac3_list[-1], label = '9+41+63')
plt.xlim(38400/16, 39100/16)
plt.ylim(-0.05, 0.6)
plt.legend()
plt.show()
printbar()

plt.figure(figsize=(20,6))
plt.subplot(1,2,1)
plt.plot( noesy_np[5,::16]-noesy_np[0,::16 ][2400:2440].min(), label = 'noesy_5_4k')
plt.plot(  scale_9*fac1_list[5], label = '9')
plt.plot(  scale_41*fac2_list[5], label = '41')
plt.plot(  scale_63*fac3_list[5], label = '63')
plt.xlim(38400/16, 39100/16)
plt.ylim(-0.05, 0.6)
plt.legend()
plt.subplot(1,2,2)
plt.plot( noesy_np[5,::16]-noesy_np[0,::16 ][2400:2440].min(), label = 'noesy_5_4k')
plt.plot(  scale_9*fac1_list[5] + scale_41*fac2_list[5]+scale_63*fac3_list[5], label = '9+41+63')
plt.xlim(38400/16, 39100/16)
plt.ylim(-0.05, 0.6)
plt.legend()
plt.show()
printbar()

plt.figure(figsize=(20,6))
plt.subplot(1,3,1)
plt.plot(fac1_list, label = '9')
plt.title(f'9@{np.round(h1_sw[2420], 2)}')
plt.subplot(1,3,2)
plt.plot(fac2_list, label = '41')
plt.title(f'41@{np.round(h1_sw[2427], 2)}')
plt.subplot(1,3,3)
plt.plot(fac3_list, label = '63')
plt.title(f'63@{np.round(h1_sw[2419], 2)}')
printbar()
plt.legend()
plt.show()

y_arr, x_arr

plt.figure(figsize=(20,6))
plt.subplot(1,3,1)
plt.plot(fac1_list/fac1_list[0], label = '9')
plt.plot(all_peak_int[5]/all_peak_int[5][0],  label = '9 HSQC')
plt.title(f'9@{np.round(h1_sw[2420], 2)}')
plt.legend()
plt.subplot(1,3,2)
plt.plot(fac2_list/fac2_list[0], label = '41')
plt.title(f'41@{np.round(h1_sw[2427], 2)}')
plt.plot(all_peak_int[12]/all_peak_int[12][0],  label = '41 HSQC')
plt.legend()
plt.subplot(1,3,3)
plt.plot(fac3_list/fac3_list[0], label = '63')
plt.plot(all_peak_int[45]/all_peak_int[45][0],  label = '63 HSQC')
plt.title(f'63@{np.round(h1_sw[2419], 2)}')
printbar()
plt.legend()
plt.show()


In [ ]:
df_new = pd.DataFrame({
    "scale_9": fac1_list,
    "scale_41": fac2_list,
    "scale_63": fac3_list,
})
df_old = pd.read_csv(factor_path)          # read existing file
df_final = pd.concat([df_old, df_new], axis=1)  # horizontal concatenation
df_final.to_csv(factor_path, index=False)   #

print("✅ Columns appended successfully！")

df_HSQC_new = pd.DataFrame({
    "scale_9": all_peak_int[5],
    "scale_41": all_peak_int[12],
    "scale_63": all_peak_int[45],
})
df_HSQC_old = pd.read_csv(factor_HSQC_path)          # read existing file
df_HSQC_final = pd.concat([df_HSQC_old, df_HSQC_new], axis=1)  # horizontal concatenation
df_HSQC_final.to_csv(factor_HSQC_path, index=False)   #

print("✅ HSQCColumns appended successfully！")


### 11c. 4.05-4.25 ppm Region (C13=18, 34)

Two-component fit for peaks in the 4.05-4.25 ppm range.

In [ ]:
scale_18 = scale_all_np.reshape(128, -1)[18]
scale_29 = scale_all_np.reshape(128, -1)[29]
scale_34 = scale_all_np.reshape(128, -1)[35]

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.contour(hsqc_4k_128_2D, edlev)
plt.xlim(2120, 2210)
plt.ylim(10, 40)
plt.subplot(1,2,2)
plt.contour(h1_sw,c13_sw, hsqc_4k_128_2D, edlev)
plt.xlim(h1_sw[2120], h1_sw[2210])
plt.ylim(c13_sw[10], c13_sw[40])
printbar()
plt.show()

peak_windows = [
    slice(2190, 2204),   # scale_18
    slice(2178, 2188)    # scale_34
]

def extract_peaks(data, windows):
    return np.concatenate([data[w] for w in windows])

scales = [scale_18, scale_34]
A = np.array([extract_peaks(s, peak_windows) for s in scales]).T

weights = np.ones(A.shape[0])
W = np.diag(weights)
A_weighted = W @ A

bounds = (0.0, np.inf)

fac1_list, fac2_list = [], []  #, fac3_list , []

for i in noesy_np[:, ::16]:
    b = extract_peaks(i-i[2160:2220].min(), peak_windows)
    b_weighted = W @ b
    res = lsq_linear(A_weighted, b_weighted, bounds=bounds, method="bvls")
    coeffs = res.x

    fac1_list.append(coeffs[0])
    fac2_list.append(coeffs[1])

printbar()
print(fac1_list, len(fac1_list))


In [ ]:
plt.figure(figsize=(20,6))
plt.subplot(1,2,1)
plt.plot( noesy_np[0,::16 ]-noesy_np[0,::16 ][2160:2220].min(), label = 'noesy_0_4k')
plt.plot(  scale_18*fac1_list[0], label = '18')
plt.plot(  scale_34*fac2_list[0], label = '34')
plt.xlim(34000/16, 36000/16)
plt.ylim(-0.05, 0.4)
plt.legend()
plt.subplot(1,2,2)
plt.plot( noesy_np[0,::16 ]-noesy_np[0,::16 ][2160:2220].min(), label = 'noesy_0_4k')
plt.plot(  scale_18*fac1_list[0] +scale_34*fac2_list[0], label = '18+34')
plt.xlim(34000/16, 36000/16)
plt.ylim(-0.05, 0.4)
plt.legend()
plt.show()

plt.figure(figsize=(20,6))
plt.subplot(1,2,1)
plt.plot( noesy_np[-1,::16]-noesy_np[0,::16 ][2160:2220].min(), label = 'noesy_-1_4k')
plt.plot(  scale_18*fac1_list[-1], label = '18')
plt.plot(  scale_34*fac2_list[-1], label = '34')
plt.xlim(34000/16, 36000/16)
plt.ylim(-0.05, 0.4)
plt.legend()
plt.subplot(1,2,2)
plt.plot( noesy_np[-1,::16 ]-noesy_np[0,::16 ][2160:2220].min(), label = 'noesy_-1_4k')
plt.plot(  scale_18*fac1_list[-1] +scale_34*fac2_list[-1], label = '18+34')
plt.xlim(34000/16, 36000/16)
plt.ylim(-0.05, 0.4)
plt.legend()
printbar()
plt.show()

plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.plot(fac1_list/fac1_list[0], label = '18')
plt.plot(all_peak_int[31]/all_peak_int[31][0], label = '18 HSQC')
plt.title(f'18@{np.round(h1_sw[2198], 2)}ppm')
plt.legend()
plt.subplot(1,3,2)
plt.plot(fac2_list/fac2_list[0], label = '34')
plt.plot(all_peak_int[37]/all_peak_int[37][0], label = '34 HSQC')
plt.title(f'34@{np.round(h1_sw[2184], 2)}ppm')
plt.legend()
printbar()
plt.show()


In [ ]:
df_new = pd.DataFrame({
    "scale_18": fac1_list,
    "scale_34": fac2_list,
})
df_old = pd.read_csv(factor_path)          # read existing file
df_final = pd.concat([df_old, df_new], axis=1)  # horizontal concatenation
df_final.to_csv(factor_path, index=False)   #

print("✅ Columns appended successfully！")

df_HSQC_new = pd.DataFrame({
    "scale_18": all_peak_int[31],
    "scale_34": all_peak_int[37],
})
df_HSQC_old = pd.read_csv(factor_HSQC_path)          # read existing file
df_HSQC_final = pd.concat([df_HSQC_old, df_HSQC_new], axis=1)  # horizontal concatenation
df_HSQC_final.to_csv(factor_HSQC_path, index=False)   #

print("✅ HSQCColumns appended successfully！")


### 11d. 2.85-2.95 ppm Region (C13=77, 81, 92, 96)

Four-component fit with inequality constraints (c81 >= c92, c81 >= c77)
to resolve the crowded anomeric region.

In [ ]:
scale_77 = scale_all_np.reshape(128, -1)[77]
scale_81 = scale_all_np.reshape(128, -1)[81]
scale_92 = scale_all_np.reshape(128, -1)[92]
scale_96 = scale_all_np.reshape(128, -1)[96]

plt.figure(figsize=(10,6))
plt.subplot(1,2,1)
plt.contour(hsqc_4k_128_2D, edlev)
plt.xlim(2860, 2940)
plt.ylim(75, 105)
plt.subplot(1,2,2)
plt.contour(h1_sw,c13_sw, hsqc_4k_128_2D, edlev)
plt.xlim(h1_sw[2860], h1_sw[2940])
plt.ylim(c13_sw[75], c13_sw[105])
plt.show()

peak_windows = [
    slice(2901, 2931),   # scale_81
    slice(2914, 2922),   # scale_92
    slice(2904, 2920),   # scale_96
    slice(2917, 2936),   # scale_77
]

def extract_peaks(data, windows):
    return np.concatenate([data[w] for w in windows])

scales = [ scale_81, scale_92, scale_96, scale_77 ]
A = np.array([extract_peaks(s, peak_windows) for s in scales]).T

weights = np.ones(A.shape[0])
W = np.diag(weights)
A_weighted = W @ A

C = np.array([
    [1, -1,  0,  0],   # c0 - c1 ≥ 0
    [1,  0,  0, -1]    # c0 - c3 ≥ 0
])
d = np.zeros(2)  # constraint RHS is ≥ 0

A_aug = np.vstack([A_weighted, C])
b_aug = np.hstack([np.zeros(A_weighted.shape[0]), d])

bounds = (0.0, np.inf)

fac1_list, fac2_list, fac3_list, fac4_list = [], [], [], []

for i in noesy_np[:, ::16]:
    b = extract_peaks(i - i[2880:2960].min(), peak_windows)
    b_weighted = W @ b

    b_total = np.hstack([b_weighted, d])

    res = lsq_linear(A_aug, b_total, bounds=bounds, method="bvls")
    coeffs = res.x

    fac1_list.append(coeffs[0])
    fac2_list.append(coeffs[1])
    fac3_list.append(coeffs[2])
    fac4_list.append(coeffs[3])

printbar()
print(fac1_list, len(fac1_list))


In [ ]:
plt.figure(figsize=(20,6))
plt.subplot(1,2,1)
plt.plot( noesy_np[0, ::16 ]-noesy_np[0, ::16 ][2880:2960].min(), label = 'noesy_0_4k')
plt.plot( scale_81*fac1_list[0], label = '81')
plt.plot( scale_92*fac2_list[0], label = '92')
plt.plot( scale_96*fac3_list[0], label = '96')
plt.plot( scale_77*fac4_list[0], label = '77')
plt.xlim(46000/16 , 47500/16 )
plt.ylim(-0.05, 1)
plt.legend()
plt.subplot(1,2,2)
plt.plot( noesy_np[0, ::16 ]-noesy_np[0, ::16 ][2880:2960].min(), label = 'noesy_0_4k')
plt.plot( scale_81*fac1_list[0] + scale_92*fac2_list[0]+scale_96*fac3_list[0]+scale_77*fac4_list[0], label = '81+92+96+77')  #+scale_102*fac4_list[0]

errors = np.sum((scale_81*fac1_list[0] + scale_92*fac2_list[0]+scale_96*fac3_list[0] - noesy_np[0, ::16 ]+noesy_np[0, ::16 ][2880:2960].min()) ** 2 ) #+scale_102*fac4_list[0]
plt.xlim(46000/16 , 47500/16 )
plt.ylim(-0.05, 1)
plt.legend()
print(errors)
plt.show()

plt.figure(figsize=(20,6))
plt.subplot(1,2,1)
plt.plot(h1_sw, noesy_np[0, ::16]-noesy_np[0, ::16 ][2880:2960].min(), label='noesy_0_4k')
plt.plot(h1_sw,scale_81 * fac1_list[0], label='81')
plt.plot(h1_sw,scale_92 * fac2_list[0], label='92')
plt.plot(h1_sw,scale_96 * fac3_list[0], label='96')
plt.plot(h1_sw,scale_77 * fac4_list[0], label='77')
plt.xlim(h1_sw[2875], h1_sw[2969])
plt.ylim(-0.05, 1)
plt.legend()

plt.subplot(1,2,2)
plt.plot(h1_sw,noesy_np[0, ::16]-noesy_np[0, ::16 ][2880:2960].min(), label='noesy_0_4k')

fit_spectrum = scale_81*fac1_list[0] + scale_92*fac2_list[0] + scale_96*fac3_list[0]+scale_77*fac4_list[0] #+ scale_102*fac4_list[0]
plt.plot(h1_sw,fit_spectrum, label='81+92+96')

errors = np.sum((fit_spectrum - noesy_np[0, ::16] ) ** 2)
print("✅ Fit MSE (MSE) =", errors)  #

plt.xlim(h1_sw[2875], h1_sw[2969])
plt.ylim(-0.05, 1)
plt.legend()
plt.show()

plt.figure(figsize=(20,6))
plt.subplot(1,2,1)
plt.plot( noesy_np[-1, ::16 ]-noesy_np[-1, ::16 ][2880:2960].min(), label = 'noesy_-1_4k')
plt.plot(  scale_81*fac1_list[-1], label = '81')
plt.plot(  scale_92*fac2_list[-1], label = '92')
plt.plot(  scale_96*fac3_list[-1], label = '96')
plt.plot(  scale_77*fac1_list[-1], label = '77')
plt.xlim(46000/16 , 47500/16 )
plt.ylim(-0.05, 1)
plt.legend()
plt.subplot(1,2,2)
plt.plot( noesy_np[-1, ::16 ]-noesy_np[-1, ::16 ][2880:2960].min(), label = 'noesy_-1_4k')
plt.plot(  scale_81*fac1_list[-1] + scale_92*fac2_list[-1]+scale_96*fac3_list[-1]+scale_77*fac4_list[-1], label = '81+92+96')

errors = np.sum((scale_81*fac1_list[-1] + scale_92*fac2_list[-1]+scale_96*fac3_list[-1] - noesy_np[-1, ::16 ]+noesy_np[-1, ::16 ][2880:2960].min()) ** 2 )
plt.xlim(46000/16 , 47500/16 )
plt.ylim(-0.05, 1)
plt.legend()
print(errors)
printbar()
plt.show()

plt.figure(figsize=(12,4))
plt.subplot(1,4,1)
plt.plot(fac1_list, label = '81')
plt.title(f'81@{np.round(h1_sw[2920], 2)}ppm')
plt.subplot(1,4,2)
plt.plot(fac2_list, label = '92')
plt.title(f'92@{np.round(h1_sw[2917], 2)}ppm')
plt.subplot(1,4,3)
plt.plot(fac3_list, label = '96')
plt.title(f'96@{np.round(h1_sw[2910], 2)}ppm')
plt.subplot(1,4,4)
plt.plot(fac4_list, label = '77')
plt.title(f'77@{np.round(h1_sw[2924], 2)}ppm')

plt.legend()
printbar()
plt.show()

plt.figure(figsize=(12, 4))
plt.subplot(1,4,1)
plt.plot(fac1_list/fac1_list[0], label = '81')
plt.plot(all_peak_int[0]/all_peak_int[0][0], label = '81 HSQC')
plt.title(f'81@{np.round(h1_sw[2920], 2)}ppm')
plt.legend()
plt.subplot(1,4,2)
plt.plot(fac2_list/fac2_list[0], label = '92')
plt.plot(all_peak_int[19]/all_peak_int[19][0], label = '92 HSQC')
plt.title(f'92@{np.round(h1_sw[2917], 2)}ppm')
plt.legend()
plt.subplot(1,4,3)
plt.plot(fac3_list/fac3_list[0], label = '96')
plt.plot(all_peak_int[6]/all_peak_int[6][0], label = '96 HSQC')
plt.title(f'96@{np.round(h1_sw[2910], 2)}ppm')
plt.legend()
plt.subplot(1,4,4)
plt.plot(fac4_list/fac4_list[0], label = '77')
plt.plot(all_peak_int[20]/all_peak_int[20][0], label = '77 HSQC')
plt.title(f'77@{np.round(h1_sw[2924], 2)}ppm')
plt.legend()
printbar()
plt.show()


In [ ]:
df_new = pd.DataFrame({
    "scale_81": fac1_list,
    "scale_92": fac2_list,
    "scale_96": fac3_list,
    "scale_77": fac4_list
})

df_old = pd.read_csv(factor_path)
df_total = pd.concat([df_old, df_new], axis=1)
df_total.to_csv(factor_path, index=False)

df_HSQC_new = pd.DataFrame({
    "scale_81": all_peak_int[0],
    "scale_92": all_peak_int[19],
    "scale_96": all_peak_int[6],
    "scale_77": all_peak_int[20],
})
df_HSQC_old = pd.read_csv(factor_HSQC_path)          # read existing file
df_HSQC_final = pd.concat([df_HSQC_old, df_HSQC_new], axis=1)  # horizontal concatenation
df_HSQC_final.to_csv(factor_HSQC_path, index=False)   #

print("✅ HSQCColumns appended successfully！")


## 12. Results Export

All fitted coefficients and HSQC intensities are exported to CSV files
in the `factors/` directory for downstream analysis.

In [ ]:
printbar()
print("="*60)
print("Analysis Complete")
print("="*60)
print(f"\nscale_all_np shape: {scale_all_np.shape}")
